In [5]:
import json
from kafka import KafkaConsumer
from dataclasses import dataclass

In [6]:
@dataclass()
class Ride:
    PULocationID: int
    DOLocationID: int
    trip_distance: float
    total_amount: float
    tpep_pickup_datetime: int  # epoch milliseconds

In [7]:
def ride_deserializer(data):
    json_str = data.decode('utf-8')
    ride_dict = json.loads(json_str)
    return Ride(**ride_dict)

In [8]:
test_bytes = json.dumps({
    'PULocationID': 186,
    'DOLocationID': 79,
    'trip_distance': 1.72,
    'total_amount': 17.31,
    'tpep_pickup_datetime': 1730429702000
}).encode('utf-8')

ride_deserializer(test_bytes)
# Ride(PULocationID=186, DOLocationID=79, trip_distance=1.72,
#      total_amount=17.31, tpep_pickup_datetime=1730429702000)

Ride(PULocationID=186, DOLocationID=79, trip_distance=1.72, total_amount=17.31, tpep_pickup_datetime=1730429702000)

In [9]:
server = 'localhost:9092'
topic_name = 'rides'

consumer = KafkaConsumer(
    topic_name,
    bootstrap_servers=[server],
    auto_offset_reset='earliest',
    group_id='rides-console',
    value_deserializer=ride_deserializer
)

In [11]:
next(consumer)

ConsumerRecord(topic='rides', partition=0, leader_epoch=1, offset=1, timestamp=1773860541334, timestamp_type=0, key=None, value=Ride(PULocationID=1, DOLocationID=10, trip_distance=20, total_amount=30, tpep_pickup_datetime=1761956005000), headers=[], checksum=None, serialized_key_size=-1, serialized_value_size=119, serialized_header_size=-1)

In [1]:
import psycopg2

conn = psycopg2.connect(
    host='localhost',
    port=5432,
    database='postgres',
    user='postgres',
    password='postgres'
)
conn.autocommit = True
cur = conn.cursor()

In [12]:
from datetime import datetime

print(f"Listening to {topic_name} and writing to PostgreSQL...")

count = 0
for message in consumer:
    ride = message.value
    pickup_dt = datetime.fromtimestamp(ride.lpep_pickup_datetime / 1000)
    dropoff_dt = datetime.fromtimestamp(ride.lpep_dropoff_datetime / 1000)
    cur.execute(
        """INSERT INTO processed_events
           (PULocationID, DOLocationID, trip_distance, total_amount, tip_amount, pickup_datetime, dropoff_datetime, passenger_count)
           VALUES (%s, %s, %s, %s, %s, %s, %s, %s)""",
        (ride.PULocationID, ride.DOLocationID,
         ride.trip_distance, ride.total_amount,ride.tip_amount, pickup_dt)
    )
    count += 1
    if count % 100 == 0:
        print(f"Inserted {count} rows...")

consumer.close()
cur.close()
conn.close()

Listening to rides...
Received: PU=43, DO=186, distance=1.68, amount=$22.15, pickup=2025-11-01 05:43:25
Received: PU=142, DO=237, distance=2.28, amount=$24.94, pickup=2025-11-01 06:19:07
Received: PU=163, DO=238, distance=2.7, amount=$25.62, pickup=2025-11-01 05:37:19
Received: PU=138, DO=261, distance=12.87, amount=$86.14, pickup=2025-11-01 05:30:00
Received: PU=138, DO=37, distance=8.4, amount=$48.65, pickup=2025-11-01 05:48:50
Received: PU=90, DO=100, distance=0.85, amount=$16.45, pickup=2025-11-01 05:51:11
Received: PU=142, DO=170, distance=3.01, amount=$25.85, pickup=2025-11-01 05:37:31
Received: PU=237, DO=144, distance=3.82, amount=$57.54, pickup=2025-11-01 06:16:52
Received: PU=162, DO=161, distance=0.89, amount=$12.95, pickup=2025-11-01 06:26:59
Received: PU=234, DO=162, distance=2.28, amount=$38.68, pickup=2025-11-01 05:40:43

... received 10 messages so far (stopping after 10 for demo)
